# Module 4 — Mô hình XGBoost (Cửa hàng × Nhóm hàng × Ngày)

Dự báo `so_luong` bằng XGBoost trên cùng bảng, cùng 14 đặc trưng và cùng mốc chia với các mô hình khác. Ba cột `_code` được khai báo là biến phân loại; không thêm thông tin mới vào bảng của Module 3.

- **Đầu vào:** `outputs/daily_store_category.csv` + `tao_dac_trung` (lag, rolling dời 1 ngày, lịch, mã cửa hàng/nhóm hàng/vùng).
- **So sánh công bằng:** `chia_tap`, `danh_gia`, `bang_baseline`. Tối ưu theo MAE trên Validation; early stopping trên Validation; train lại trên 15/01–28/02; chấm Test tháng 3.
- **Đầu ra** `outputs/xgb/`: `xgb_tuning.csv`, `xgb_best_params.json`, `xgb_metrics.csv`, `xgb_model.joblib`, `xgb_predictions.csv`, `xgb_feature_importance.*`, `xgb_error_by_category.csv`, `xgb_error_by_store.csv`, `xgb_thuc_te_vs_du_bao.png`.

**Cấu hình** (`.env` ở gốc repo, mẫu `.env.example`). **Chạy trên Google Colab:** upload `module3_preprocess.py` và `.env` cùng thư mục với notebook, rồi Runtime → Run all.

In [ ]:
import os, sys, json, time, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import ParameterSampler

sys.path.append("../module3")
from module3_preprocess import (OUT_DIR as OUT, SEED, doc_bang_ngay, tao_dac_trung, chia_tap, danh_gia,
                                bang_baseline, loi_theo_nhom, nop_ket_qua, tai_ve)

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
OUT_X = f"{OUT}/xgb"; os.makedirs(OUT_X, exist_ok=True)
CAT = ["Mã cửa hàng_code", "Nhóm hàng_code", "Vùng_code"]
MAX_ITER, STOPPING = 1500, 75
daily = doc_bang_ngay()
print(f"Bảng mô hình: {len(daily):,} dòng")

## 1. Đặc trưng và chia tập

Dùng `tao_dac_trung`, `chia_tap`, `bang_baseline` của Module 3. XGBoost nhận ba cột mã dưới dạng `category` (`enable_categorical`); mọi mô hình vẫn dùng cùng các cột đầu vào. Một quan sát tại ngày t chỉ dùng số bán từ các ngày trước t.

In [ ]:
d, FEATS = tao_dac_trung(daily)
assert set(CAT).issubset(FEATS)
kieu = {c: pd.CategoricalDtype(sorted(d[c].dropna().unique())) for c in CAT}

def as_x(df):
    """Cùng FEATS; ba cột mã thành category để XGBoost không coi mã là số thứ tự."""
    x = df[FEATS].copy()
    for c, dt in kieu.items():
        x[c] = x[c].astype(dt)
    return x

fit, val, tr, te = chia_tap(d)
base_val = pd.DataFrame(bang_baseline(val)).T
rows = bang_baseline(te)
print("Baseline trên Validation:"); display(base_val)
print("Baseline trên Test:"); display(pd.DataFrame(rows).T)

## 2. Tối ưu tham số trên Validation

Thử 4 hàm mất mát (`reg:squarederror`, `reg:absoluteerror`, `count:poisson`, `reg:tweedie`) × 3 bộ tham số = 12 cấu hình. Early stopping 75 vòng trên Validation, tối đa 1.500 vòng; chọn cấu hình có **MAE Validation** thấp nhất. Lưu bảng thử nghiệm và cấu hình chọn (kèm số vòng) vào `outputs/xgb/`.

In [ ]:
SPACE = {"max_depth": [4, 6, 8], "min_child_weight": [20, 50, 100],
         "learning_rate": [0.03, 0.05, 0.1], "reg_lambda": [1.0, 3.0, 10.0]}
OBJECTIVES = [
    {"objective": "reg:squarederror", "eval_metric": "rmse"},
    {"objective": "reg:absoluteerror", "eval_metric": "mae"},
    {"objective": "count:poisson", "eval_metric": "poisson-nloglik"},
    {"objective": "reg:tweedie", "eval_metric": "tweedie-nloglik@1.5", "tweedie_variance_power": 1.5},
]
sampled = list(ParameterSampler(SPACE, n_iter=3, random_state=SEED))
params = [{**o, **p} for o in OBJECTIVES for p in sampled]
CO_DINH = {"subsample": 0.8, "colsample_bytree": 0.8, "tree_method": "hist"}

def dung_xgb(p, n_estimators, early=False):
    cb = [xgb.callback.EarlyStopping(rounds=STOPPING, save_best=True, maximize=False)] if early else None
    return xgb.XGBRegressor(**p, **CO_DINH, n_estimators=n_estimators, random_state=SEED, n_jobs=-1,
                            enable_categorical=True, importance_type="gain", verbosity=0, callbacks=cb)

log = []
for i, p in enumerate(params):
    t0 = time.time()
    m = dung_xgb(p, MAX_ITER, early=True)
    m.fit(as_x(fit), fit["so_luong"], eval_set=[(as_x(val), val["so_luong"])], verbose=False)
    s = danh_gia(val["so_luong"], np.clip(m.predict(as_x(val)), 0, None))
    vong = int(m.best_iteration) + 1
    log.append({"id": i, **p, "so_vong": vong, **s, "giây": round(time.time() - t0, 1)})
    print(f"{i + 1:2}/{len(params)} MAE={s['MAE']:.4f} Bias={s['Bias']:+.1%} vòng={vong:4} {p}", flush=True)
tuning = pd.DataFrame(log).sort_values("MAE").reset_index(drop=True)
tuning.to_csv(f"{OUT_X}/xgb_tuning.csv", index=False, encoding="utf-8-sig")
chon = params[int(tuning.loc[0, "id"])]
best = {**chon, **CO_DINH, "n_estimators": int(tuning.loc[0, "so_vong"]), "early_stopping_rounds": STOPPING}
with open(f"{OUT_X}/xgb_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best, f, indent=2, ensure_ascii=False)
print("Tốt nhất:", best)
display(tuning.head(5))

## 3. Train lại trên toàn bộ Train và chấm Test

Train lại trên 15/01–28/02 theo đúng số vòng đã chọn; Test là tháng 3. Dự báo âm được cắt về 0.

In [ ]:
xgb_model = dung_xgb(chon, best["n_estimators"], early=False)
xgb_model.fit(as_x(tr), tr["so_luong"], verbose=False)
te["du_bao_xgb"] = np.clip(xgb_model.predict(as_x(te)), 0, None)
rows["XGBoost"] = danh_gia(te["so_luong"], te["du_bao_xgb"])
result = pd.DataFrame(rows).T
result["WAPE so với MA7"] = result["WAPE"] / result.loc["MA7", "WAPE"] - 1
result.to_csv(f"{OUT_X}/xgb_metrics.csv", encoding="utf-8-sig")
joblib.dump(xgb_model, f"{OUT_X}/xgb_model.joblib", compress=3)
display(result)

## 4. Đặc trưng quan trọng và lỗi theo nhóm hàng / cửa hàng / ngày

Importance của XGBoost là tổng gain trên các nhánh cây; phần trăm trong báo cáo được chuẩn hóa từ giá trị này.

In [ ]:
imp = pd.Series(xgb_model.feature_importances_, index=FEATS, name="importance").sort_values()
imp.to_csv(f"{OUT_X}/xgb_feature_importance.csv", header=["importance"], encoding="utf-8-sig")
fig, ax = plt.subplots(figsize=(7, 5)); ax.barh(imp.index, imp.values, color="#eda100")
ax.set_title("XGBoost — mức quan trọng đặc trưng (gain)"); fig.tight_layout()
fig.savefig(f"{OUT_X}/xgb_feature_importance.png", dpi=150); plt.show()

err_cat = loi_theo_nhom(te, "du_bao_xgb", "Nhóm hàng")
err_store = loi_theo_nhom(te, "du_bao_xgb", ["Mã cửa hàng", "Khu vực"])
err_cat.to_csv(f"{OUT_X}/xgb_error_by_category.csv", encoding="utf-8-sig")
err_store.to_csv(f"{OUT_X}/xgb_error_by_store.csv", encoding="utf-8-sig")
display(err_cat)
print(f"XGBoost có WAPE thấp hơn MA7 ở {(err_cat.WAPE < err_cat.WAPE_MA7).sum()}/{len(err_cat)} nhóm hàng "
      f"và {(err_store.WAPE < err_store.WAPE_MA7).sum()}/{len(err_store)} cửa hàng")

day = te.groupby("Ngày")[["so_luong", "du_bao_xgb", "tb_7"]].sum()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day.index, day["so_luong"], color="#1f2a2a", lw=2, label="Thực tế")
ax.plot(day.index, day["du_bao_xgb"], color="#eda100", lw=2, label="XGBoost")
ax.plot(day.index, day["tb_7"], color="#898781", lw=1.5, ls="--", label="MA7")
ax.set_title("Tổng số lượng tháng 3 — thực tế và dự báo"); ax.legend(); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig(f"{OUT_X}/xgb_thuc_te_vs_du_bao.png", dpi=150); plt.show()
te[["Mã cửa hàng", "Nhóm hàng", "Khu vực", "Vùng", "Ngày", "so_luong", "du_bao_xgb", "lag_1", "lag_7", "tb_7"]] \
    .to_csv(f"{OUT_X}/xgb_predictions.csv", index=False, encoding="utf-8-sig")

## 5. Kết quả và thảo luận

*Số liệu từ lần chạy ngày 12/10/2026 trên bảng mô hình chưa lọc (giữ bán buôn, online, trả lại) — cùng bảng với Ridge, Poisson, Random Forest và LightGBM. Test tháng 3 gồm 67.146 ô.*

**Tối ưu tham số.** Trong 12 cấu hình, `objective=reg:absoluteerror`, `max_depth=4`, `min_child_weight=20`, `learning_rate=0.03`, `reg_lambda=1`, `n_estimators=179` có MAE Validation thấp nhất: 0,753 so với 0,847 của MA7. Ba cấu hình tốt nhất đều dùng `reg:absoluteerror` (MAE 0,753–0,767) và đều dự báo thiếu khoảng 19–20% trên Validation. Tweedie tốt nhất đạt 0,856; squared error và Poisson quanh 0,89–0,91.

| Cấu hình (3 tốt nhất theo MAE Val) | MAE Val | RMSE Val | WAPE Val | Bias Val |
|---|---:|---:|---:|---:|
| absoluteerror · depth 4 · child 20 · lr 0,03 · 179 vòng | **0,753** | **8,564** | **0,666** | −19,6% |
| absoluteerror · depth 4 · child 50 · lr 0,05 · 121 vòng | 0,758 | 8,610 | 0,671 | **−19,2%** |
| absoluteerror · depth 8 · child 50 · lr 0,03 · 178 vòng | 0,767 | 8,620 | 0,679 | −19,5% |
| *MA7 (tham chiếu)* | *0,847* | *8,474* | *0,749* | *+11,2%* |

**Kết quả trên Test (tháng 3, 67.146 ô):**

| Mô hình | MAE | RMSE | WAPE | R² | Bias |
|---|---:|---:|---:|---:|---:|
| Naive (lag 1) | 0,887 | 6,675 | 0,802 | −0,142 | −0,9% |
| Seasonal Naive (lag 7) | 0,908 | 6,728 | 0,821 | −0,160 | +1,6% |
| MA7 | 0,744 | **4,975** | 0,674 | **0,365** | **−0,2%** |
| **XGBoost** | **0,700** | 5,527 | **0,634** | 0,217 | −30,5% |

**Thảo luận.**

1. **XGBoost thắng MA7 về MAE và WAPE, thua về RMSE, R² và Bias.** MAE 0,700 so với 0,744 (giảm 5,9%), WAPE 0,634 so với 0,674 (cũng giảm 5,9%). RMSE 5,527 so với 4,975; R² 0,217 so với 0,365. Thắng WAPE ở 22/23 nhóm hàng và 124/130 cửa hàng.
2. **Thiếu tổng lượng.** Bias −30,5%: tổng dự báo 51.541 so với 74.196 sản phẩm thực tế; trung bình 1.663 sản phẩm/ngày so với 2.393 (MA7 2.388). Ở 278 ô bán ≥ 20 sản phẩm/ngày (thực tế 13.570), XGBoost dự báo 4.250 — thấp hơn cả MA7 (10.904). Hàm mất mát trị tuyệt đối ưu tiên trung vị khi khoảng 70% ô bằng 0, nên dự báo bị kéo về thấp.
3. **Giảm lỗi ở ô không bán, kém ở ô có bán.** 46.477 ô bán 0 có MAE 0,148 so với 0,241 của MA7; các ô có bán có MAE 1,942 so với 1,876. Điện thoại thắng MA7 (WAPE 0,547 so với 0,559); Hcare thắng (0,531 so với 0,552); Phụ kiện — nhóm lớn thứ hai — là nhóm duy nhất thua (0,516 so với 0,507).
4. **Đặc trưng quan trọng (gain):** lag 1 (39,8%), trung bình 14 ngày (20,4%), trung bình 7 ngày (17,3%). Bảy đặc trưng lượng bán gần đây cộng 86,5%; `ngay_trong_thang` 5,2%; ba cột mã cộng 4,8% (`Vùng_code` gần 0). Khác LightGBM, nơi mã cửa hàng và `ngay_trong_thang` chiếm phần lớn gain.
5. **So với các mô hình đã có** (cùng bảng, đặc trưng, mốc chia): XGBoost cùng kiểu với LightGBM (loss L1, thắng MAE/WAPE, thiếu hàng) nhưng kém LightGBM về MAE (0,700 so với 0,678) và vẫn thiếu nhiều hơn ngưỡng (Bias −30,5% so với −37,3% của LightGBM). Random Forest (MAE 0,757; Bias +0,8%) và Poisson (MAE 0,799; Bias +5,7%) giữ |Bias| ≤ 10% nhưng chưa thắng MA7 về MAE. Ridge MAE 0,802, Bias +10,7%. Theo tiêu chí đề xuất (MAE thấp nhất trong các mô hình có |Bias| ≤ 10%), XGBoost không đạt.

**Hướng cải thiện:** cấu hình tốt nhất nằm ở biên `learning_rate` nhỏ nhất và `min_child_weight` nhỏ nhất của lưới; đổi loss hoặc thêm ràng buộc |Bias| khi chọn cấu hình sẽ quan trọng hơn là chỉ hạ MAE Validation.

**Giới hạn:** Validation 15–28/02 không đại diện cho tháng 3 (Bias −19,6% trên Validation, −30,5% trên Test); bảng chưa lọc bán buôn/online; 12 cấu hình.

## Nộp kết quả vào báo cáo và tải về (Colab)

`nop_ket_qua` lọc các kết quả đưa vào báo cáo sang `bao_cao/du_lieu/xgb/`. Sau đó chạy `python module5/tao_bao_cao.py` để cập nhật `bao_cao/BAO_CAO.md` rồi commit `bao_cao/` (trên Colab: giải nén `bao_cao_xgb.zip` vào `bao_cao/du_lieu/xgb/` trong repo trước).

In [ ]:
nop_ket_qua("xgb", val)
tai_ve(OUT_X, "module4_xgb_outputs")